# Generate SG/PL Prefix Pairs from MultiBLiMP

For each MultiBLiMP SV-# item, generate a counterpart prefix with the opposite subject number. The verb continuations from MultiBLiMP swap roles automatically.

**Example** (original is SG):
- `prefix_sg = "The dog"` → `continuation_sg_verb = "bites the cat."` ✓
- `prefix_pl = "The dogs"` → `continuation_sg_verb = "bites the cat."` ✗ (new ungrammatical)
- `prefix_pl = "The dogs"` → `continuation_pl_verb = "bite the cat."` ✓ (new grammatical)

**For feature discovery:** compare SAE activations on `prefix_sg` vs `prefix_pl` at the pre-verb position.

In [11]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

from src.data.number_pairs import (
    load_sv_number_items,
    generate_prefix_pairs,
    reconstruct_sentences,
    summarize_generation,
    save_pairs,
)
from dotenv import load_dotenv
load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY before running.\n"
    "  export OPENAI_API_KEY='sk-...'"
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
LANGUAGES = ["eng", "deu", "spa"]
MODEL = "gpt-5"
DELAY = 0.01
MAX_ITEMS = None

## 1. Load & Preview MultiBLiMP SV-# Items

In [13]:
source_data = {}
for lang in LANGUAGES:
    df = load_sv_number_items(lang)
    if MAX_ITEMS:
        df = df.head(MAX_ITEMS)
    source_data[lang] = df
    print(f"\n{lang}: {len(df)} items")
    print(f"  grammatical_feature: {df['grammatical_feature'].value_counts().to_dict()}")
    print(f"  Sample sen:     {df['sen'].iloc[0]}")
    print(f"  Sample wrong:   {df['wrong_sen'].iloc[0]}")

2026-04-12 22:09:12,366 | src.data.multiblimp | INFO | Using cached MultiBLiMP data for eng
2026-04-12 22:09:12,507 | src.data.multiblimp | INFO | Filtered 770 -> 317 number-agreement pairs (lang=eng)
2026-04-12 22:09:12,524 | src.data.number_pairs | INFO | Loaded 300 SV-# (SV order) items for eng

eng: 300 items
  grammatical_feature: {'PL': 200, 'SG': 100}
  Sample sen:     The below site from the US government verifies this-it 's the same for Canada ,too.....
  Sample wrong:   The below site from the US government verify this-it 's the same for Canada ,too.....
2026-04-12 22:09:12,532 | src.data.multiblimp | INFO | Using cached MultiBLiMP data for deu
2026-04-12 22:09:12,595 | src.data.multiblimp | INFO | Filtered 2298 -> 800 number-agreement pairs (lang=deu)
2026-04-12 22:09:12,599 | src.data.number_pairs | INFO | Loaded 400 SV-# (SV order) items for deu

deu: 400 items
  grammatical_feature: {'SG': 200, 'PL': 200}
  Sample sen:     Die Grafschaft, die sich zwischen Rabnitzbach und

## 2. Generate Flipped Prefixes

The LLM rewrites each prefix with the opposite subject number. Continuations (verb forms) are taken from MultiBLiMP and role-swapped automatically.

Set `MAX_ITEMS = 5` above for a test run first.

In [ ]:
import importlib
import src.data.number_pairs as _np_mod
importlib.reload(_np_mod)
from src.data.number_pairs import (
    call_openai, build_messages, parse_response, extract_continuations,
)

test_row = source_data["spa"].iloc[1]
conts = extract_continuations(test_row)
messages = build_messages(
    str(test_row["prefix"]).rstrip(),
    str(test_row["child"]).strip(),
    "singular", "plural", "spa",
    continuation_sg_verb=conts[0],
    continuation_pl_verb=conts[1],
)
print("=== MESSAGES ===")
for m in messages:
    print(f"[{m['role']}] {m['content'][:120]}...")
print()

raw = call_openai(messages, model=MODEL)
print(parse_response(raw))

### Alternative: local Qwen3-30B

Run the same generation pipeline with Qwen3-30B loaded locally via transformers (same pattern as notebook 01). Needs a GPU with ~40 GB VRAM in bf16 (or use 4-bit quantisation on smaller GPUs).

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_ID = "Qwen/Qwen3-30B-A3B"
QWEN_DEVICE = "auto"
QWEN_DTYPE = torch.bfloat16

print(f"Loading {QWEN_ID} ...")
qwen_tok = AutoTokenizer.from_pretrained(QWEN_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_ID, torch_dtype=QWEN_DTYPE, device_map=QWEN_DEVICE,
)
qwen_model.eval()
print(f"Loaded on {next(qwen_model.parameters()).device}")


def call_qwen(messages: list[dict]) -> str:
    text = qwen_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = qwen_tok(text, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=True, temperature=0.3, top_p=0.9,
        )
    response = qwen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    # strip any residual <think> blocks
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    return response.strip()


# quick sanity check
test = call_qwen([{"role": "user", "content": 'Reply with only: {"status": "ok"}'}])
print("Qwen test:", test)

In [ ]:
results_qwen = {}
for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"  Generating flipped prefixes for: {lang} (Qwen3-30B)")
    print(f"{'='*60}")
    df_in = source_data[lang].head(MAX_ITEMS) if MAX_ITEMS else source_data[lang]
    result_df = generate_prefix_pairs(
        df_in, lang_code=lang, llm_fn=call_qwen, delay=0,
        tag="raw_qwen",
    )
    results_qwen[lang] = result_df
    print(f"  → {len(result_df)} items processed")

for lang in LANGUAGES:
    print(f"\n── {lang} ──")
    for k, v in summarize_generation(results_qwen[lang]).items():
        print(f"  {k}: {v}")

### Base: OpenAI API-based generation

In [ ]:
results = {}
for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"  Generating flipped prefixes for: {lang}")
    print(f"{'='*60}")
    df_in = source_data[lang].head(MAX_ITEMS) if MAX_ITEMS else source_data[lang]
    result_df = generate_prefix_pairs(
        df_in,
        lang_code=lang,
        model=MODEL,
        delay=DELAY,
        tag="raw",
    )
    results[lang] = result_df
    print(f"  → {len(result_df)} items processed")

## 3. Validation Summary

In [ ]:
for lang in LANGUAGES:
    print(f"\n── {lang} ──")
    summary = summarize_generation(results[lang])
    for k, v in summary.items():
        print(f"  {k}: {v}")

## 4. Review Prefix Pairs & Reconstructed Sentences

In [ ]:
review_lang = "eng"
df = results[review_lang]
valid = df[df["auto_valid"]].copy()

print(f"Valid prefix pairs for {review_lang}: {len(valid)}\n")
for _, row in valid.head(15).iterrows():
    sents = reconstruct_sentences(row)
    print(f"  SG prefix: {row['prefix_sg']}")
    print(f"  PL prefix: {row['prefix_pl']}")
    print(f"    ✓ SG correct:   {sents['sg_correct']}")
    print(f"    ✗ SG incorrect: {sents['sg_incorrect']}")
    print(f"    ✓ PL correct:   {sents['pl_correct']}")
    print(f"    ✗ PL incorrect: {sents['pl_incorrect']}")
    print()

## 5. Save

Save auto-valid pairs.

In [ ]:
KEEP_COLS = [
    "language", "prefix_sg", "prefix_pl",
    "continuation_sg_verb", "continuation_pl_verb",
    "original_number", "has_attractor", "distance",
    "source_idx",
]

for lang in LANGUAGES:
    valid = results[lang][results[lang]["auto_valid"]].copy()
    cols = [c for c in KEEP_COLS if c in valid.columns]
    valid = valid[cols].reset_index(drop=True)
    valid.insert(0, "pair_id", [f"{lang}_{i:04d}" for i in range(len(valid))])

    path = save_pairs(valid, lang, tag="generated")
    print(f"{lang}: saved {len(valid)} pairs → {path}")

## 6. Export `same_verb` tables (fixed continuation, flipped prefixes)

For each generated pair, write two evaluation rows:
- `target_number=SG`: hold SG continuation fixed, compare SG prefix (good) vs PL prefix (bad)
- `target_number=PL`: hold PL continuation fixed, compare PL prefix (good) vs SG prefix (bad)

This is format for testing the same verb continuation under different prefixes.

In [ ]:
from src.data.number_pairs import export_same_verb_prefix_pairs

# from src.data.number_pairs import export_multiblimp_extended,
from src.data.number_pairs import PAIRS_DIR

LANGUAGES = ["eng", "deu", "spa"]

generated_pairs = {
    lang: pd.read_json(PAIRS_DIR / f"{lang}_generated.json")
    for lang in LANGUAGES
}
SAME_VERB_PATHS = export_same_verb_prefix_pairs(LANGUAGES, generated_pairs)

for lang, path in sorted(SAME_VERB_PATHS.items()):
    n = len(pd.read_csv(path, sep="\t"))
    print(f"{lang}: {n} rows → {path}")

Final files used in evaluation are found in `data/number_pairs/{LANG}_same_verb.tsv`.